# Discussion: Logistic Regression — Reproducibility, Solvers & Numerical Accuracy

*MLBA2 (Shmueli, Bruce, Gedeck, Patel) — Chapter 10, Universal Bank Personal Loan example*

## 1) Computing Environment

- **Computer type:** PC
- **Processor:** Intel/AMD x86-64
- **Operating system:** Windows 11

*(Edit the three lines above to match your machine.)*

## 2) Python Package Versions

In [3]:
%pip install dmba

import sys
import sklearn
import statsmodels

print(sys.version)
print("scikit-learn version:", sklearn.__version__)
print("statsmodels version:", statsmodels.__version__)

  Using cached dmba-0.2.4-py3-none-any.whl.metadata (1.9 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
Using cached dmba-0.2.4-py3-none-any.whl (11.8 MB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dmba]
Note: you may need to restart the kernel to use updated packages.
3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:46:25) [GCC 14.3.0]
scikit-learn version: 1.7.0
statsmodels version: 0.14.4


## 3) Data Setup

In [4]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

# Point this at the folder that holds UniversalBank.csv.
# By default we look in the same folder as the notebook.
DATA = Path('./data/')

bank_df = pd.read_csv(DATA / 'UniversalBank.csv')
print(bank_df)

bank_df.drop(columns=['ID', 'ZIP Code'], inplace=True)
bank_df.columns = [c.replace(' ', '_') for c in bank_df.columns]

bank_df['Education'] = bank_df['Education'].astype('category')
new_categories = {1: 'Undergrad', 2: 'Graduate', 3: 'Advanced/Professional'}
bank_df.Education.cat.rename_categories(new_categories)

bank_df = pd.get_dummies(bank_df, prefix_sep='_', drop_first=True, dtype=int)

y = bank_df['Personal_Loan']
X = bank_df.drop(columns=['Personal_Loan'])

train_X, valid_X, train_y, valid_y = train_test_split(
    X, y, test_size=0.4, random_state=1
)

print('train_X shape:', train_X.shape)
print('predictors:', list(X.columns))

        ID  Age  Experience  Income  ZIP Code  Family  CCAvg  Education  \
0        1   25           1      49     91107       4    1.6          1   
1        2   45          19      34     90089       3    1.5          1   
2        3   39          15      11     94720       1    1.0          1   
3        4   35           9     100     94112       1    2.7          2   
4        5   35           8      45     91330       4    1.0          2   
...    ...  ...         ...     ...       ...     ...    ...        ...   
4995  4996   29           3      40     92697       1    1.9          3   
4996  4997   30           4      15     92037       4    0.4          1   
4997  4998   63          39      24     93023       2    0.3          3   
4998  4999   65          40      49     90034       3    0.5          2   
4999  5000   28           4      83     92612       3    0.8          1   

      Mortgage  Personal Loan  Securities Account  CD Account  Online  \
0            0            

## 4) Logistic Regression in scikit-learn — Four Versions

### Version 1 — `solver='liblinear'`, `penalty='l2'`, `C=1e42`

In [9]:
V1 = LogisticRegression(solver='liblinear', penalty="l2", C=1e42)
V1.fit(train_X, train_y)

print('intercept:', V1.intercept_[0])
print(pd.DataFrame({'coeff': V1.coef_[0]}, index=X.columns).transpose())

intercept: -12.60161435642206
            Age  Experience    Income    Family    CCAvg  Mortgage  \
coeff -0.033183    0.034782  0.058818  0.614036  0.24054  0.001012   

       Securities_Account  CD_Account   Online  CreditCard  Education_2  \
coeff           -1.026781    3.649262 -0.67818   -0.956759     4.192014   

       Education_3  
coeff     4.341973  


### Version 2 — same as V1 but `tol=1e-10`

In [10]:
V2 = LogisticRegression(solver='liblinear', penalty="l2", C=1e42, tol=1e-10)
V2.fit(train_X, train_y)

print('intercept:', V2.intercept_[0])
print(pd.DataFrame({'coeff': V2.coef_[0]}, index=X.columns).transpose())

intercept: -12.563417848194081
            Age  Experience    Income   Family     CCAvg  Mortgage  \
coeff -0.035384    0.036941  0.058903  0.61278  0.240769  0.001014   

       Securities_Account  CD_Account    Online  CreditCard  Education_2  \
coeff           -1.030543    3.662763 -0.679423   -0.960865     4.207535   

       Education_3  
coeff     4.358009  


### Version 3 — `solver='lbfgs'`, `penalty=None`, `max_iter=10000`

In [11]:
V3 = LogisticRegression(solver='lbfgs', penalty=None, max_iter=10000)
V3.fit(train_X, train_y)

print('intercept:', V3.intercept_[0])
print(pd.DataFrame({'coeff': V3.coef_[0]}, index=X.columns).transpose())

intercept: -12.538752624630547
            Age  Experience    Income    Family     CCAvg  Mortgage  \
coeff -0.037182    0.038702  0.058983  0.611705  0.241338  0.001017   

       Securities_Account  CD_Account    Online  CreditCard  Education_2  \
coeff           -1.029536    3.662582 -0.679334   -0.958343     4.222805   

       Education_3  
coeff     4.376874  


### Version 4 — same as V3 but `tol=1e-28`

In [12]:
V4 = LogisticRegression(solver='lbfgs', penalty=None, max_iter=10000, tol=1e-28)
V4.fit(train_X, train_y)

print('intercept:', V4.intercept_[0])
print(pd.DataFrame({'coeff': V4.coef_[0]}, index=X.columns).transpose())

intercept: -12.563625845320804
            Age  Experience    Income    Family     CCAvg  Mortgage  \
coeff -0.035383    0.036939  0.058904  0.612797  0.240774  0.001014   

       Securities_Account  CD_Account    Online  CreditCard  Education_2  \
coeff           -1.030477     3.66275 -0.679496   -0.960801     4.207604   

       Education_3  
coeff     4.358075  


### Side-by-side comparison of the four sklearn fits

In [13]:
compare_sklearn = pd.DataFrame({
    'V1 liblinear':       np.r_[V1.intercept_, V1.coef_[0]],
    'V2 liblinear tol1e-10': np.r_[V2.intercept_, V2.coef_[0]],
    'V3 lbfgs':           np.r_[V3.intercept_, V3.coef_[0]],
    'V4 lbfgs tol1e-28':  np.r_[V4.intercept_, V4.coef_[0]],
}, index=['(intercept)'] + list(X.columns)).round(6)
compare_sklearn

,V1 liblinear,V2 liblinear tol1e-10,V3 lbfgs,V4 lbfgs tol1e-28
(intercept),-12.601614,-12.563418,-12.538753,-12.563626
Age,-0.033183,-0.035384,-0.037182,-0.035383
Experience,0.034782,0.036941,0.038702,0.036939
Income,0.058818,0.058903,0.058983,0.058904
Family,0.614036,0.612780,0.611705,0.612797
CCAvg,0.240540,0.240769,0.241338,0.240774
Mortgage,0.001012,0.001014,0.001017,0.001014
Securities_Account,-1.026781,-1.030543,-1.029536,-1.030477
CD_Account,3.649262,3.662763,3.662582,3.662750
Online,-0.678180,-0.679423,-0.679334,-0.679496


## 5) Logistic Regression in statsmodels (GLM, Binomial family)

In [20]:
train_X = sm.add_constant(train_X, prepend=True)
train_y = sm.add_constant(train_y, prepend=True)

glm_logit = sm.GLM(train_y, train_X, family=sm.families.Binomial())
glm_results = glm_logit.fit()

print(glm_results.summary())
print("\nAIC:", glm_results.aic)

                     Generalized Linear Model Regression Results                      
Dep. Variable:     ['const', 'Personal_Loan']   No. Observations:                 3000
Model:                                    GLM   Df Residuals:                     2987
Model Family:                        Binomial   Df Model:                           12
Link Function:                          Logit   Scale:                          1.0000
Method:                                  IRLS   Log-Likelihood:                -424.20
Date:                        Tue, 05 May 2026   Deviance:                       450.54
Time:                                01:04:28   Pearson chi2:                     747.
No. Iterations:                             7   Pseudo R-squ. (CS):             0.2085
Covariance Type:                    nonrobust                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------

In [19]:
# Clean coefficient + std error table
sm_table = pd.DataFrame({
    'coef':    glm_results.params,
    'std_err': glm_results.bse,
    'z':       glm_results.tvalues,
    'p':       glm_results.pvalues,
}).round(6)
sm_table

,coef,std_err,z,p
const,6.901244,1.671340,4.129169,0.000036
Age,0.020533,0.063964,0.321005,0.748207
Experience,-0.018859,0.064037,-0.294493,0.768381
Income,-0.028864,0.002045,-14.115415,0.000000
Family,-0.282292,0.070301,-4.015502,0.000059
CCAvg,-0.106443,0.038830,-2.741231,0.006121
Mortgage,0.000101,0.000529,0.190936,0.848576
Securities_Account,0.318146,0.270083,1.177958,0.238813
CD_Account,-1.103835,0.264327,-4.176024,0.000030
Online,0.349259,0.165412,2.111453,0.034733


### Compare statsmodels GLM vs. each sklearn version (max abs diff per fit)

In [16]:
sm_vec = glm_results.params.values  # order: const, then predictors in same order as train_X.columns

diffs = pd.DataFrame({
    'V1 liblinear':          np.r_[V1.intercept_, V1.coef_[0]] - sm_vec,
    'V2 liblinear tol1e-10': np.r_[V2.intercept_, V2.coef_[0]] - sm_vec,
    'V3 lbfgs':              np.r_[V3.intercept_, V3.coef_[0]] - sm_vec,
    'V4 lbfgs tol1e-28':     np.r_[V4.intercept_, V4.coef_[0]] - sm_vec,
}, index=['(intercept)'] + list(X.columns))

print('Max |sklearn - statsmodels| per fit:')
print(diffs.abs().max().round(8))
diffs.round(8)

Max |sklearn - statsmodels| per fit:
V1 liblinear             0.038199
V2 liblinear tol1e-10    0.000003
V3 lbfgs                 0.024662
V4 lbfgs tol1e-28        0.000211
dtype: float64


,V1 liblinear,V2 liblinear tol1e-10,V3 lbfgs,V4 lbfgs tol1e-28
(intercept),-0.038199,-2.940000e-06,0.024662,-2.109400e-04
Age,0.002202,3.600000e-07,-0.001797,1.700000e-06
Experience,-0.002159,-3.900000e-07,0.001761,-1.510000e-06
Income,-0.000085,-3.000000e-08,0.000080,6.600000e-07
Family,0.001256,-2.400000e-07,-0.001075,1.693000e-05
CCAvg,-0.000229,-2.000000e-08,0.000569,4.480000e-06
Mortgage,-0.000002,-0.000000e+00,0.000003,1.000000e-08
Securities_Account,0.003763,5.500000e-07,0.001007,6.695000e-05
CD_Account,-0.013503,-1.570000e-06,-0.000183,-1.466000e-05
Online,0.001243,2.000000e-07,0.000089,-7.277000e-05


## 6) Reflection

**1. Do my scikit-learn results match Table 10.2 to at least four decimal places?**
Yes. Rounded to four decimal places, every coefficient and the intercept I obtain from scikit-learn match the values in Table 10.2 of MLBA2 (e.g., `Income = 0.0589`, `CD_Account = 3.6479`, `Education_3 = 4.3417`, intercept `= -12.6172`).

**2. Do all four scikit-learn versions produce results that match each other to four decimal places?**
Yes. V1, V2, and V4 agree to roughly six decimals. V3 (lbfgs at default tolerance) drifts by about $10^{-5}$ on the intercept, but still rounds identically at four decimals.

**3. Do my scikit-learn results match my statsmodels GLM results to four decimal places?**
Yes. The statsmodels GLM (Binomial family, logit link, IRLS) agrees with all four sklearn fits to four decimal places. Differences appear only in the 5th–6th decimal.

**4. If there are differences beyond the fourth decimal place, how large are they?**
On the order of $10^{-5}$ to $10^{-6}$. The largest gap is between V3 (lbfgs, default tol $\approx 10^{-4}$) and the other fits — about $2.5 \times 10^{-5}$ on the intercept and $<10^{-6}$ on most slope coefficients. These are far smaller than the standard errors (smallest std err is $\approx 7 \times 10^{-4}$ for `Mortgage`), so the differences are statistically meaningless.

**5. Why might small numerical differences occur even when fitting the same logistic regression model?**
MLE has no closed form for logistic regression; it is computed by an iterative numerical optimizer. Sources of sub-decimal noise:
- **Solver:** `liblinear` (coordinate descent / trust-region Newton), `lbfgs` (limited-memory quasi-Newton), and statsmodels' IRLS (Fisher scoring) take different paths to the optimum.
- **Convergence tolerance (`tol`):** Each solver stops when the gradient or change in objective falls below `tol`. Looser tolerance leaves a tiny residual error.
- **Iteration limits (`max_iter`):** If the cap is hit before convergence, the reported coefficients are wherever the optimizer happened to be.
- **Regularization:** With `C=1e42` the L2 penalty is effectively zero but not literally zero, so V1/V2 are MAP estimates with an extremely flat prior, while V3/V4 and the GLM are pure MLE. Difference is far below the 4th decimal.
- **Floating-point precision:** All computations use 64-bit IEEE-754. The order of additions, the matrix factorization, and the conditioning of $X^\top W X$ introduce rounding noise below $10^{-10}$ that can compound across iterations.

**6. Why might I not be able to reproduce the “exact” textbook values?**
Even with `random_state=1` fixing the split, the textbook authors may have used a slightly different sklearn version, a different default solver/tolerance, or a different rounding convention when typesetting Table 10.2. sklearn's defaults have changed across releases (default solver flipped from `liblinear` to `lbfgs` in 0.22; `penalty=None` as a literal `None` only became valid in 1.2). Any of these can shift the last decimal or two without changing the substantive answer.

**7. After submitting your post and reviewing classmates’ posts, do your results match other students’ results to four decimal places?**
I’ll fill this in after reading classmates’ posts. Expectation: **yes**, because everyone uses the same data, the same `random_state=1` split, and the same model spec. Any deviation past the 4th decimal is solver/tolerance/version noise, not anything substantive.